# SQL Joins & Multi-Table Analysis

This notebook demonstrates relational database join semantics and validation frameworks:
1. **LEFT JOIN with Row Count Validation** (understanding 1-to-many row multiplication).
2. **Detecting Unmatched Keys & Orphaned Records** (`IS NULL` filtering).
3. **Comparing Join Types** (`INNER JOIN`, `LEFT JOIN`, `FULL OUTER JOIN`).
4. **Multi-Table Joins (4 Tables)** connecting customers, orders, order_items, and products.
5. **Data Lineage Validation & Duplication Prevention**.

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

db_path = '../analytics.db'
if not os.path.exists(db_path):
    db_path = 'analytics.db'

engine = create_engine(f'sqlite:///{db_path}')
print(f"Connected to relational database: {db_path}")

## Task 1: LEFT JOIN with Row Count Validation

In [2]:
def load_query(query_name):
    path = f'../queries/{query_name}.sql'
    if not os.path.exists(path):
        path = f'queries/{query_name}.sql'
    with open(path, 'r') as f:
        return f.read()

customers_count = pd.read_sql("SELECT COUNT(*) as ct FROM customers", engine).iloc[0]['ct']
left_sql = load_query('left_join_validation')
joined_df = pd.read_sql(left_sql, engine)

print(f"Base Customers Count : {customers_count}")
print(f"Aggregated Result    : {len(joined_df)} rows")
print(joined_df.head())

## Task 2: Detect Unmatched Keys & Orphaned Records

In [3]:
unmatched_cust_sql = load_query('unmatched_customers')
no_orders_df = pd.read_sql(unmatched_cust_sql, engine)
print(f"Customers with NO Orders: {len(no_orders_df)}")
print(no_orders_df.head(3))

unmatched_ord_sql = load_query('unmatched_orders')
orphaned_df = pd.read_sql(unmatched_ord_sql, engine)
print(f"\nOrphaned Orders (Invalid customer_id): {len(orphaned_df)}")
print(orphaned_df.head(3))

## Task 3: Compare Join Types (INNER vs. LEFT vs. FULL)

In [4]:
join_comp_sql = load_query('join_types_comparison')
comp_df = pd.read_sql(join_comp_sql, engine)
print("Join Types Row Count Comparison:")
print(comp_df)

## Task 4: Multi-Table Join & Duplication Assertions

In [5]:
multi_sql = load_query('multi_table_join')
multi_df = pd.read_sql(multi_sql, engine)
print(f"4-Table Join Result Set: {len(multi_df):,} rows")
print(multi_df.head())